# India-Only Baseline — YOLOv11s on RDD2022 India Subset

**Goal:** Establish a clean India-only baseline with YOLOv11s before applying any balancing or augmentation.

**Split:** 70/20/10 (Train: 5394 | Val: 1541 | Test: 771)

**Inputs required in Kaggle side panel:**
- `India_RDD` — the India RDD2022 dataset (`India/train/images/`, `India/train/labels/`)
- `Baseline_India_Yolov11s` — the split txt files (`_Baseline_India_train.txt`, `_Baseline_India_val.txt`, `_Baseline_India_test.txt`)


## 1. Environment Setup

In [ ]:
# GPU check
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# Install Ultralytics
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

In [ ]:
# Disk space check
!df -h /kaggle/working /kaggle/input 2>/dev/null

## 2. Paths and data.yaml

In [ ]:
import os

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

# Dataset root — txt files contain relative paths like India/train/images/India_000001.jpg
# so this must be the parent of India/
DATASET_ROOT = "/kaggle/input/datasets/musfiraaarif/india-rdd"

# Split txt files
TRAIN_TXT = "/kaggle/input/datasets/musfiraaarif/baseline-india-yolov11s/_Baseline_India_train.txt"
VAL_TXT   = "/kaggle/input/datasets/musfiraaarif/baseline-india-yolov11s/_Baseline_India_val.txt"
TEST_TXT  = "/kaggle/input/datasets/musfiraaarif/baseline-india-yolov11s/_Baseline_India_test.txt"

# Verify everything exists
for label, p in [
    ("DATASET_ROOT", DATASET_ROOT),
    ("TRAIN_TXT",    TRAIN_TXT),
    ("VAL_TXT",      VAL_TXT),
    ("TEST_TXT",     TEST_TXT),
]:
    status = "OK  " if os.path.exists(p) else "MISSING"
    print(f"{status}  {label}: {p}")

# Sanity check: verify first path in train txt resolves correctly
with open(TRAIN_TXT) as f:
    first = f.readline().strip()
full = os.path.join(DATASET_ROOT, first)
print(f"\nFirst train image path check:")
print(f"  Relative: {first}")
print(f"  Full:     {full}")
print(f"  Exists:   {os.path.exists(full)}")

# Convert split lists (relative OR absolute entries) to absolute paths
def make_absolute(src, dst, root):
    with open(src) as f:
        lines = [l.strip() for l in f if l.strip()]
    abs_lines = [os.path.join(root, l) for l in lines]
    with open(dst, "w") as f:
        f.write("\n".join(abs_lines))
    print(f"Written {len(abs_lines)} lines to {dst}; first exists: {os.path.exists(abs_lines[0])}")
    return dst

ABS_TRAIN_TXT = make_absolute(TRAIN_TXT, f"{WORK_DIR}/train_baseline.txt", DATASET_ROOT)
ABS_VAL_TXT   = make_absolute(VAL_TXT,   f"{WORK_DIR}/val_baseline.txt",   DATASET_ROOT)
ABS_TEST_TXT  = make_absolute(TEST_TXT,  f"{WORK_DIR}/test_baseline.txt",  DATASET_ROOT)


In [ ]:
# Generate data.yaml in-session
data_yaml = f"""# India-only baseline — generated in-session
path: /
train: {ABS_TRAIN_TXT}
val:   {ABS_VAL_TXT}

names:
  0: D00
  1: D10
  2: D20
  3: D40
"""

DATA_YAML_PATH = os.path.join(WORK_DIR, "data_india_baseline.yaml")
with open(DATA_YAML_PATH, "w") as f:
    f.write(data_yaml)

print("data.yaml written to:", DATA_YAML_PATH)
print()
print(data_yaml)

## 3. Dataset Verification

In [ ]:
# Count images and label class distribution
from collections import defaultdict

LABELS_DIR = os.path.join(DATASET_ROOT, "India", "train", "labels")
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}

counts = defaultdict(int)
label_files = [f for f in os.listdir(LABELS_DIR) if f.endswith(".txt")]

for fname in label_files:
    with open(os.path.join(LABELS_DIR, fname)) as f:
        for line in f:
            line = line.strip()
            if line:
                cls = int(line.split()[0])
                counts[cls] += 1

print(f"Total label files: {len(label_files)}")
print("\nClass distribution (full India train set):")
for cls in sorted(counts):
    print(f"  {CLASS_NAMES[cls]}: {counts[cls]}")

majority = max(counts.values())
print("\nImbalance ratios (majority / class):")
for cls in sorted(counts):
    print(f"  {CLASS_NAMES[cls]}: {majority / counts[cls]:.1f}:1")

In [ ]:
# Check for missing labels (images with no corresponding label file)
IMAGES_DIR = os.path.join(DATASET_ROOT, "India", "train", "images")

missing = []
for img in os.listdir(IMAGES_DIR):
    if not img.endswith(".jpg"):
        continue
    lbl = img.replace(".jpg", ".txt")
    if not os.path.exists(os.path.join(LABELS_DIR, lbl)):
        missing.append(img)

print(f"Images without labels: {len(missing)}")
if missing:
    print("First 5:", missing[:5])

## 4. Training

India-only baseline. No balancing, no augmentation changes, no architecture changes.

- Model: YOLOv11s (pretrained COCO)
- Epochs: 150 (patience 50), batch 32, image size 640
- SGD, lr0 = 0.01, momentum 0.937, weight decay 5e-4, 3 warmup epochs
- val=True (held-out val split used during training)

These settings are identical to the balanced run and match `models/baseline/best.pt`.


In [ ]:
from ultralytics import YOLO

RUN_NAME = "india_baseline_yolov11s"

model = YOLO("yolo11s.pt")  # downloads COCO-pretrained weights

results = model.train(
    data=DATA_YAML_PATH,
    epochs=150,
    imgsz=640,
    batch=32,
    optimizer="SGD",
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    device=0,
    val=True,
    patience=50,
    save=True,
    save_period=5,
    project="/kaggle/working/runs",
    name=RUN_NAME,
    exist_ok=True,
)

FINAL_RUN_NAME = RUN_NAME
print("Training complete.")
print("Run dir:", f"/kaggle/working/runs/{FINAL_RUN_NAME}")


## 5. Resume (if session expired)

If your Kaggle session ended before training finished:
1. Download `runs/india_baseline_yolov11s/weights/last.pt` from the Output tab
2. Upload it as a new Kaggle dataset
3. Start a fresh session, re-run cells 1–4, then run the cell below

In [ ]:
# # --- RESUME — only run if previous session expired ----------------------
# LAST_PT = "/kaggle/input/YOUR-CHECKPOINT-DATASET/last.pt"  # update path
# resume_model = YOLO(LAST_PT)
# resume_model.train(resume=True)

## 6. Evaluation on Test Set

In [ ]:
# Generate a test yaml pointing at the held-out test split
test_yaml = f"""path: /
train: {ABS_TRAIN_TXT}
val:   {ABS_TEST_TXT}

names:
  0: D00
  1: D10
  2: D20
  3: D40
"""

TEST_YAML_PATH = os.path.join(WORK_DIR, "data_india_test.yaml")
with open(TEST_YAML_PATH, "w") as f:
    f.write(test_yaml)
print("Test yaml written to:", TEST_YAML_PATH)

In [ ]:
# Evaluate best.pt on held-out test set
BEST_PT = f"/kaggle/working/runs/{FINAL_RUN_NAME}/weights/best.pt"

eval_model = YOLO(BEST_PT)
metrics = eval_model.val(
    data=TEST_YAML_PATH,
    imgsz=640,
    batch=16,
    device=0,
    split="val",  # Ultralytics reads the 'val' key from yaml
)

# Extract and print F1 per class
P = metrics.box.p   # precision per class [D00, D10, D20, D40]
R = metrics.box.r   # recall per class

print("\n=== India Baseline Results ===")
print(f"{'Class':<8} {'P':>8} {'R':>8} {'F1':>8}")
print("-" * 36)
class_names = ["D00", "D10", "D20", "D40"]
for i, name in enumerate(class_names):
    p, r = P[i], R[i]
    f1 = 2 * p * r / (p + r + 1e-8)
    print(f"{name:<8} {p:>8.4f} {r:>8.4f} {f1:>8.4f}")

# Overall F1 (macro average)
import numpy as np
f1_all = [2*P[i]*R[i]/(P[i]+R[i]+1e-8) for i in range(len(P))]
print("-" * 36)
print(f"{'Mean':<8} {np.mean(P):>8.4f} {np.mean(R):>8.4f} {np.mean(f1_all):>8.4f}")
print(f"\nmAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 7. Save Outputs

In [ ]:
import shutil

RUN_DIR = f"/kaggle/working/runs/{FINAL_RUN_NAME}"

# List run contents
print("Run directory contents:")
for root, dirs, files in os.walk(RUN_DIR):
    level = root.replace(RUN_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

# Zip for download
zip_path = shutil.make_archive(
    base_name=f"/kaggle/working/{FINAL_RUN_NAME}",
    format="zip",
    root_dir="/kaggle/working/runs",
    base_dir=FINAL_RUN_NAME,
)
print("\nZipped to:", zip_path)